In [1]:
from satellite_RFI.src.satellite_sims import satellite_sim as ss
import time
import pickle
import astropy.units as u
from datetime import datetime
import tqdm
import os


import scipy as sp
import numpy as np
import pandas as pd
import scipy.optimize as opt
import matplotlib.pyplot as plt

In [3]:
obs_time_input=None#'2021 9 30 20 06 36'
# fname = '1554156377'
fname = '1551055211'


"""
Establishing the file name
"""
if obs_time_input!=None:
    obs_time_in=[int(x) for x in obs_time_input.split()]
    obs_time = datetime(obs_time_in[0], obs_time_in[1], obs_time_in[2], obs_time_in[3], obs_time_in[4], obs_time_in[5])
    dt = obs_time.strftime('%Y-%m-%d %H:%M:%S')
    fname = int((obs_time - datetime(1970, 1, 1)).total_seconds())
    print ("File name to be used is: "+str(fname))

else:
    print ("File name to be used is: "+str(fname))
    dt = (datetime.utcfromtimestamp(float(fname)).strftime('%Y-%m-%d %H:%M:%S'))
    print ("Date of observation date: "+str(dt))


##   
    
katdal_info = pickle.load(open('/idia/projects/hi_im/brandon/1551055211/'+str(fname)+'_katdal_info.p', 'rb'), encoding='latin1')



info = [katdal_info[i] for i in katdal_info.keys()]

nd_s0=katdal_info['nd_s0']
nd_s0_coords=katdal_info['nd_s0_coords']
frequency=katdal_info['frequency']
fs=1000
fe=1500

data_save='../data_test/'+str(fname)+'/pickle_info/'
if os.path.exists(data_save)==False:
    os.mkdir(data_save)
    
data_mkat = '../../../Observation_results/Untangle/'+str(fname)+'/'


deg = 2    
nearby_constellation_path = 'nearby_satellite_mask/nearby_satellite_close_angle_'+str(deg)+'_fill.p'

folder = '2022_03_20'


File name to be used is: 1551055211
Date of observation date: 2019-02-25 00:40:11


### Full data

In [ ]:
cons = ['GPS', 'SBAS', 'GAL', 'BDS', 'GLO', 'IRNSS']
bias = np.ones(len(cons))

sat_fmask = ss(file_name=fname, sats_only=None, data_loc=data_mkat, sat_loc=data_mkat,
            survey_info=[nd_s0, nd_s0_coords, frequency], sat_info='../Satellite_simulations/Satellite_Catalogue/table3B_satellite_v3-1_reduced_2.csv',
            plots_loc='data_test_plots/'+str(fname)+'/'+folder+'/',
            sat_beam='emss_beam_r', frequency_range=[1000,1500], constellations=cons, nearby_satellites=nearby_constellation_path)

sat_funmask = ss(file_name=fname, sats_only=None, data_loc=data_mkat, sat_loc=data_mkat,
            survey_info=[nd_s0, nd_s0_coords, frequency], sat_info='../Satellite_simulations/Satellite_Catalogue/table3B_satellite_v3-1_reduced_2.csv',
            plots_loc='data_test_plots/'+str(fname)+'/'+folder+'/',
            sat_beam='emss_beam_r', frequency_range=[1000,1500], constellations=cons, nearby_satellites=nearby_constellation_path)



In [ ]:
file_m = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_masked_full.p', 'rb'))
file_um = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_no_masked_full.p', 'rb'))

In [ ]:
# Masked 
sat_fmask.excecute(a_param=file_m['best-fit'], obs_time_start=file_m['time'][0], obs_time_end=file_m['time'][-1], obs_frequency_start=1100, obs_frequency_end=1350, 
            file_bias_choice=bias, add_sub=[1, 1], band_lvl=[None, None], bandsize=None)

masked_sim_sat = np.ma.array(data=sat_fmask.simulation_TOD_slice, mask=sat_fmask.mask_nearby_satellites_slice.T).T
masked_data_sat = np.ma.array(data=sat_fmask.calibration_data_slice, mask=sat_fmask.mask_nearby_satellites_slice.T).T

masked_full = [masked_sim_sat, masked_data_sat]

# Unmasked 
sat_funmask.excecute(a_param=file_um['best-fit'], obs_time_start=file_um['time'][0], obs_time_end=file_um['time'][-1], obs_frequency_start=1100, obs_frequency_end=1350, 
            file_bias_choice=bias, add_sub=[1, 1], band_lvl=[None, None], bandsize=None)

unmasked_sim_sat =sat_funmask.simulation_TOD_slice.T
unmasked_data_sat = sat_funmask.calibration_data_slice.T

unmasked_full = [unmasked_sim_sat, unmasked_data_sat]

extent = [sat_fmask.frequency_band[sat_fmask.frequency_idx[0]], sat_fmask.frequency_band[sat_fmask.frequency_idx[1]],\
            sat_fmask.nd_s0[sat_fmask.time_idx[1]], sat_fmask.nd_s0[sat_fmask.time_idx[0]]]


waterfall = [masked_sim_sat, masked_data_sat, unmasked_sim_sat, unmasked_data_sat]
waterfall_name = ['Masked Simulation', 'Masked Data', 'Unmasked Simulation', 'Unmasked Data']


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(24,4), sharey=True)
fig.subplots_adjust()

for plot in range(4):
    ax=axs[plot]
    if plot<2:
        val_max, val_min = np.ma.max(masked_full), np.ma.min(masked_full)
    else:
        val_max, val_min = np.ma.max(unmasked_full), np.ma.min(unmasked_full)

    hb = ax.imshow(waterfall[plot], aspect='auto', extent=extent, vmax=val_max, vmin=val_min)
    ax.set_title(waterfall_name[plot])
    cbar = plt.colorbar(hb,ax=ax)
    cbar.set_label(r'(T) [K]', rotation=270, labelpad=20, y=0.45)

    if plot==0:
        ax.set_ylabel('Time [sec]')
    ax.set_xlabel('Frequency [MHz]')

    
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/waterfall_'+num+'.pdf')
plt.show()

### Full TOD cut into chunks


In [ ]:
fig, axs = plt.subplots(7, 3, figsize=(21,18), sharex=True, sharey=False)
fig.subplots_adjust()

for r_plot in range(7):
    for c_plot in range(3):
        
        extent = [sat_fmask.frequency_band[sat_fmask.frequency_idx[0]], sat_fmask.frequency_band[sat_fmask.frequency_idx[1]],\
            sat_fmask.nd_s0[100*(3*r_plot+c_plot)+100], sat_fmask.nd_s0[100*(3*r_plot+c_plot)]]
                
        
        ax=axs[r_plot, c_plot]
        
        hb=ax.imshow(masked_sim_sat[100*(3*r_plot+c_plot):100*(3*r_plot+c_plot)+100, :], aspect='auto', extent=extent)
        cbar = plt.colorbar(hb,ax=ax)
        cbar.set_label(r'(T) [K]', rotation=270, labelpad=20, y=0.45)
        
        
        ax.set_title('Chunk '+str(3*r_plot+c_plot))
        
        if 3*r_plot+c_plot > 17:
            ax.set_xlabel('Frequency')
        
        ax.set_ylabel('Time [sec]')
        
plt.tight_layout()
plt.savefig('data_test_plots/'+fname+'/'+folder+'/all_TOD_chunks_fill.pdf')
plt.show()

In [ ]:
freqs = sat_funmask.frequency_band[sat_funmask.frequency_idx[0]:sat_funmask.frequency_idx[1]]

fig, axs = plt.subplots(1, 2, figsize=(16,4), sharey=False)
fig.subplots_adjust()
for plot in range(2):
    ax=axs[plot]
 
    ax.plot(freqs, np.ma.mean(waterfall[2*plot], axis=0), label='Simulation')
    ax.plot(freqs, np.ma.mean(waterfall[2*plot+1], axis=0), label='MeerKAT data')

    ax.legend()  
    if plot<1:
        ax.set_ylabel('Temperature [K]')
    ax.set_xlabel('Frequency [MHz]')
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/average_over_time_'+num+'.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(np.arange(len(file_m['best-fit'])), file_m['best-fit'], label='Mask')
plt.plot(np.arange(len(file_m['best-fit'])), file_um['best-fit'], label='Unmask')

plt.title('All Alpha values for the ')

plt.xlabel('# Alpha')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()

### Chunck data

In [ ]:
cons = ['GPS', 'SBAS', 'GAL', 'BDS', 'GLO', 'IRNSS']
bias = np.ones(len(cons))

sat_mask = ss(file_name=fname, sats_only=None, data_loc=data_mkat, sat_loc=data_mkat,
            survey_info=[nd_s0, nd_s0_coords, frequency], sat_info='../Satellite_simulations/Satellite_Catalogue/table3B_satellite_v3-1_reduced_2.csv',
            plots_loc='data_test_plots/'+str(fname)+'/'+folder+'/',
            sat_beam='emss_beam_r', frequency_range=[1000,1500], constellations=cons, nearby_satellites=nearby_constellation_path)

sat_unmask = ss(file_name=fname, sats_only=None, data_loc=data_mkat, sat_loc=data_mkat,
            survey_info=[nd_s0, nd_s0_coords, frequency], sat_info='../Satellite_simulations/Satellite_Catalogue/table3B_satellite_v3-1_reduced_2.csv',
            plots_loc='data_test_plots/'+str(fname)+'/'+folder+'/',
            sat_beam='emss_beam_r', frequency_range=[1000,1500], constellations=cons, nearby_satellites=nearby_constellation_path)



In [ ]:
# num = 'chunk_'+str(1)  # Glo-ops
# num = 'chunk_'+str(11)  # galileo
num = 'chunk_'+str(13)  # none

chunk_file_m = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_masked_'+num+'.p', 'rb'))
chunk_file_um = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_un_masked_'+num+'.p', 'rb'))

In [ ]:
print ('Timestamp:\t'+str(np.round(chunk_file_m['time'][0],2))+'\t\t'+str(np.round(chunk_file_m['time'][-1],2)))

In [ ]:
# Masked 
sat_mask.excecute(a_param=chunk_file_m['best-fit'], obs_time_start=chunk_file_m['time'][0], obs_time_end=chunk_file_m['time'][-1], obs_frequency_start=1100, obs_frequency_end=1350, 
            file_bias_choice=bias, add_sub=[1, 1], band_lvl=[None, None], bandsize=None)

masked_sim_sat = np.ma.array(data=sat_mask.simulation_TOD_slice, mask=sat_mask.mask_nearby_satellites_slice.T).T
masked_data_sat = np.ma.array(data=sat_mask.calibration_data_slice, mask=sat_mask.mask_nearby_satellites_slice.T).T

masked = [masked_sim_sat, masked_data_sat]

# Unmasked 
sat_unmask.excecute(a_param=chunk_file_um['best-fit'], obs_time_start=chunk_file_um['time'][0], obs_time_end=chunk_file_um['time'][-1], obs_frequency_start=1100, obs_frequency_end=1350, 
            file_bias_choice=bias, add_sub=[1, 1], band_lvl=[None, None], bandsize=None)

unmasked_sim_sat =sat_unmask.simulation_TOD_slice.T
unmasked_data_sat = sat_unmask.calibration_data_slice.T

unmasked = [unmasked_sim_sat, unmasked_data_sat]


extent = [sat_unmask.frequency_band[sat_unmask.frequency_idx[0]], sat_unmask.frequency_band[sat_unmask.frequency_idx[1]],\
            sat_unmask.nd_s0[sat_unmask.time_idx[1]], sat_unmask.nd_s0[sat_unmask.time_idx[0]]]


waterfall = [masked_sim_sat, masked_data_sat, unmasked_sim_sat, unmasked_data_sat]
waterfall_name = ['Masked Simulation', 'Masked Data', 'Unmasked Simulation', 'Unmasked Data']



In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(24,6), sharey=True)
fig.subplots_adjust()

for plot in range(4):
    ax=axs[plot]
    if plot<2:
        val_max, val_min = np.ma.max(masked), np.ma.min(masked)
    else:
        val_max, val_min = np.ma.max(unmasked), np.ma.min(unmasked)

    hb = ax.imshow(waterfall[plot], aspect='auto', extent=extent, vmax=val_max, vmin=val_min)
    ax.set_title(waterfall_name[plot])
    cbar = plt.colorbar(hb,ax=ax)
    cbar.set_label(r'(T) [K]', rotation=270, labelpad=20, y=0.45)

    if plot==0:
        ax.set_ylabel('Time [sec]')
    ax.set_xlabel('Frequency [MHz]')

    
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/waterfall_'+num+'.pdf')
plt.show()

In [ ]:
freqs = sat_unmask.frequency_band[sat_unmask.frequency_idx[0]:sat_unmask.frequency_idx[1]]

fig, axs = plt.subplots(1, 2, figsize=(16,4), sharey=False)
fig.subplots_adjust()
for plot in range(2):
    ax=axs[plot]
 
    ax.plot(freqs, np.ma.mean(waterfall[2*plot], axis=0), label='Simulation')
    ax.plot(freqs, np.ma.mean(waterfall[2*plot+1], axis=0), label='MeerKAT data')

    ax.legend()  
    if plot<1:
        ax.set_ylabel('Temperature [K]')
    ax.set_xlabel('Frequency [MHz]')
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/average_over_time_'+num+'.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10,4))


plt.plot(freqs, np.ma.mean(waterfall[0]-waterfall[1], axis=0), label='Mask')
plt.plot(freqs, np.ma.mean(waterfall[2]-waterfall[3], axis=0), label='Unmask', alpha=0.4)

plt.title('Difference between the Simulation and Data')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Temperature [K]')

plt.legend()
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/difference_sim_vs_data_'+num+'.pdf')
plt.show()

### All chunk alpha data


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(chunk_file_m['best-fit'], label='Mask')
plt.plot(chunk_file_um['best-fit'], label='Unmask')

plt.title('All Alpha values for the '+num)

plt.xlabel('# Alpha')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
chunk_alpha = []

for c_i in range(22):

    num='chunk_'+str(c_i)
    chunk_file_m = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_masked_'+num+'.p', 'rb'))
    chunk_file_um = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_un_masked_'+num+'.p', 'rb'))

    chunk_alpha.append([chunk_file_m['best-fit'], chunk_file_um['best-fit']])

chunk_alpha = np.array(chunk_alpha)

In [ ]:
fig, axs = plt.subplots(7, 3, figsize=(21,18), sharex=True, sharey=True)
fig.subplots_adjust()

for r_plot in range(7):
    for c_plot in range(3):
        ax=axs[r_plot, c_plot]
        
        ax.plot(chunk_alpha[3*r_plot+c_plot, 0, :], label='Mask')
        ax.plot(chunk_alpha[3*r_plot+c_plot, 1, :], label='Unmask')
        ax.plot(file_m['best-fit'], 'k--', label='Full sim mask')
        
        ax.set_title('Chunk '+str(3*r_plot+c_plot))
        
        if 3*r_plot+c_plot > 17:
            ax.set_xlabel('Alpha #')
        ax.legend()
        
        
plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/all_chunks_.pdf')
plt.show()

### All alpha  values

In [ ]:
alpha_val_total = []
for a_i in range(21):
    alpha_val = []

    for c_i in range(22):

        num='chunk_'+str(c_i)
        chunk_file_m = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_masked_'+num+'.p', 'rb'))
        chunk_file_um = pickle.load(open('data_test/'+str(fname)+'/data_info/parallel_'+folder+'/data_test_un_masked_'+num+'.p', 'rb'))

        alpha_val.append([chunk_file_m['best-fit'][a_i], chunk_file_um['best-fit'][a_i]])


    alpha_val = np.array(alpha_val)
    alpha_val_total.append(alpha_val)
alpha_val_total=np.array(alpha_val_total)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(alpha_val_total[0,:,0], label='Mask')
plt.plot(alpha_val_total[0,:,1], label='Un-mask')

plt.title('1st Alpha')
plt.xlabel('Chunk #')
plt.ylabel('Amplitude')

plt.legend()
plt.tight_layout()
plt.savefig('data_test_plots/'+fname+'/'+folder+'/all_1st_alpha_.pdf')
plt.show()

In [ ]:
fig, axs = plt.subplots(7, 3, figsize=(21,18), sharex=True)
fig.subplots_adjust()

for r_plot in range(7):
    for c_plot in range(3):
        ax=axs[r_plot, c_plot]
        
        ax.set_title('Alpha '+str(3*r_plot+c_plot))
        ax.plot(alpha_val_total[3*r_plot+c_plot,:,0], label='Mask')
        ax.plot(alpha_val_total[3*r_plot+c_plot,:,1], label='Un-mask')
        ax.axhline(file_m['best-fit'][3*r_plot+c_plot], color='k', ls='--', label='Full data')
        
        if 3*r_plot+c_plot > 17:
            ax.set_xlabel('Chunk #')
        ax.legend()


plt.tight_layout()
# plt.savefig('data_test_plots/'+fname+'/'+folder+'/all_alpha.pdf')
plt.show()